# NB_lab_test_trends

Builds `<catalog>.gold.lab_test_trends` per the STTM *Silver to Gold (CareSync Health Network)*.

* **Grain:** one row per test type (`test_name`) per calendar month
* **Driving table:** `<catalog>.silver.lab_results` grouped by `test_name`, year/month of `test_date`
* **Joins:** optional LEFT JOIN `silver.calendar` on `date_key = yyyyMMdd(date_trunc('month', test_date))` for month/quarter names
* **Window:** `LAG(abnormal_rate_pct)` partitioned by `test_name` ordered by `year, month`
* **Filters:** none
* **Load type:** FULL - `CREATE OR REPLACE TABLE ... AS SELECT`

In [0]:
dbutils.widgets.text("catalog", "healthcare_databricks", "Catalog")
catalog = dbutils.widgets.get("catalog")

target_table = f"{catalog}.gold.lab_test_trends"
print(f"Target: {target_table}")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {target_table}
COMMENT 'Gold: one row per lab test type per month with abnormal/critical rates and month-over-month trend (STTM silver to gold)'
AS
WITH monthly AS (
  SELECT
    test_name,
    year(test_date)                                         AS year,
    month(test_date)                                        AS month,
    COUNT(*)                                                AS total_tests,
    COUNT(CASE WHEN result_status = 'Abnormal' THEN 1 END)  AS abnormal_test_count,
    COUNT(CASE WHEN result_status = 'Critical' THEN 1 END)  AS critical_test_count
  FROM {catalog}.silver.lab_results
  GROUP BY test_name, year(test_date), month(test_date)
),
rates AS (
  SELECT
    *,
    CAST(ROUND(abnormal_test_count * 100.0 / NULLIF(total_tests, 0), 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(ROUND(critical_test_count * 100.0 / NULLIF(total_tests, 0), 2) AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly
),
trend AS (
  SELECT
    *,
    LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month) AS prev_month_abnormal_rate_pct
  FROM rates
)
SELECT
  CAST(t.test_name AS STRING)                                  AS test_name,
  CAST(t.year AS INT)                                          AS year,
  CAST(t.month AS INT)                                         AS month,
  CAST(c.month_name AS STRING)                                 AS month_name,
  CAST(c.quarter_name AS STRING)                               AS quarter_name,
  CAST(t.total_tests AS INT)                                   AS total_tests,
  CAST(t.abnormal_test_count AS INT)                           AS abnormal_test_count,
  CAST(t.critical_test_count AS INT)                           AS critical_test_count,
  t.abnormal_rate_pct,
  t.critical_rate_pct,
  CAST(t.prev_month_abnormal_rate_pct AS DECIMAL(5,2))         AS prev_month_abnormal_rate_pct,
  CAST(t.abnormal_rate_pct - t.prev_month_abnormal_rate_pct AS DECIMAL(6,2)) AS abnormal_rate_change_pct
FROM trend t
LEFT JOIN {catalog}.silver.calendar c
  ON c.date_key = t.year * 10000 + t.month * 100 + 1
""")

In [0]:
# Grain check: (test_name, year, month) must be unique; total tests must reconcile to silver.lab_results
display(spark.sql(f"""
SELECT
  COUNT(*)                                                              AS gold_rows,
  COUNT(DISTINCT test_name, year, month)                                AS distinct_test_months,
  SUM(total_tests)                                                      AS sum_total_tests,
  (SELECT COUNT(*) FROM {catalog}.silver.lab_results)                   AS silver_lab_results,
  SUM(CASE WHEN month_name IS NULL THEN 1 ELSE 0 END)                   AS missing_calendar_join,
  SUM(CASE WHEN prev_month_abnormal_rate_pct IS NULL THEN 1 ELSE 0 END) AS first_month_rows,
  COUNT(DISTINCT test_name)                                             AS test_types
FROM {target_table}
"""))
display(spark.table(target_table).orderBy("test_name", "year", "month").limit(10))

In [0]:
# Return the gold table record count to the caller (e.g. a job or dbutils.notebook.run)
updated_df = spark.table(target_table)

record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))